# 01 · Mapeo de las 323 clases a USDA FoodData Central (API) con todos los nutrientes

Deja una tabla `clase → alimento USDA` con **todos los nutrientes por 100 g** (más porciones/gramos por medida casera), lista para estimar calorías y macros a partir de la clase y el volumen/peso.

**Idea.** El LLM **no** decide de memoria: solo elige entre candidatos devueltos por la API de USDA, así que no puede inventar ids. Un modelo (DeepSeek V4 Pro) propone el mapeo; **todas** las filas se revisan después a mano (segundo revisor) y se marcan las que necesitan revisión técnica.

| Paso | Qué hace |
|---|---|
| 1 | Carga `classes.json` (de `00_datos`) |
| 2 | Modelo A expande cada clase en consultas en inglés estilo USDA (sinónimos, estado crudo/cocido/seco) |
| 3 | `foods/search` de la API trae candidatos (Foundation, SR Legacy, FNDDS) |
| 4 | El modelo A elige **solo entre candidatos** (JSON validado) |
| 5 | Se marcan desacuerdos, `proxy`/`composite`, estado dudoso, ids inválidos |
| 6 | `review_queue.csv`: lo marcado + muestra de auditoría; se revisa a mano |
| 7 | Se descargan los nutrientes completos de los ids finales → tablas larga y ancha + tasa de error (Wilson) |

**Antes de ejecutar:** API key gratuita de USDA (<https://api.data.gov/signup/>) y API key de OpenCode Go, como secretos/variables `USDA_API_KEY` y `OPENCODE_API_KEY`. **No pegues las keys en el cuaderno ni las subas a un repositorio** (USDA desactiva las keys que aparecen públicas).

**Cuota de USDA:** 1.000 peticiones/hora por IP; si se excede, la key se bloquea 1 hora. El cuaderno gasta ~700 llamadas en total, guarda todo en caché y se detiene solo cuando queda poca cuota: basta relanzar más tarde y continúa donde quedó. En Kaggle/Colab la IP es compartida con otros usuarios, así que si ves 429 pronto, es normal esperar.

In [ ]:
import sys, os, re, json, time, math, random, hashlib, threading, unicodedata, subprocess
from pathlib import Path
from collections import Counter, defaultdict
from concurrent.futures import ThreadPoolExecutor

try: import requests
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "requests"], check=False); import requests
import numpy as np, pandas as pd
try:
    from tqdm.auto import tqdm
except ImportError:
    def tqdm(x, **kw): return x

In [ ]:
CFG = dict(
    classes_json = "classes.json",   # junto a este cuaderno; None = autodetectar
    # --- USDA API ---
    usda_key_env = "USDA_API_KEY",
    usda_base    = "https://api.nal.usda.gov/fdc/v1",
    data_types   = ["Foundation", "SR Legacy", "Survey (FNDDS)"],   # sin Branded: son por porción y de marca
    page_size = 12,          # resultados por búsqueda
    queries_per_class = 2,   # consultas de búsqueda por clase (la 1ª y la mejor alternativa)
    n_candidates = 15,       # candidatos que ve el LLM
    quota_reserve = 40,      # si quedan menos peticiones que esto, se detiene (X-RateLimit-Remaining)
    # --- LLM (OpenCode Go, endpoint compatible con OpenAI) ---
    api_key_env = "OPENCODE_API_KEY",
    base_url    = "https://opencode.ai/inference/go/openai/v1/chat/completions",
    models_url  = "https://opencode.ai/inference/go/v1/models",
    model_a = "deepseek-v4-pro",     # dos familias distintas; la celda 4 comprueba que existan en tu plan
    model_b = None,         # un solo modelo; el segundo revisor es la revisión manual (celda 7)
    temperature = 0, max_tokens = 6000, workers = 3, timeout = 180,
    # --- pipeline ---
    audit_frac = 0.10,       # fracción de lo que pasa solo que se manda a auditoría
    seed = 42,
    limit = None,            # prueba con p. ej. 15 clases antes de lanzar las 323
    reviewed_csv = "review_queue_done.csv",
    recipes_csv = "recetas.csv",     # composición de platos compuestos (ver celda 8)
    out_dir = None,
)
OUT = Path(CFG["out_dir"] or ("/kaggle/working/nutri_map" if Path("/kaggle/working").exists() else "./nutri_map"))
OUT.mkdir(parents=True, exist_ok=True)
random.seed(CFG["seed"]); np.random.seed(CFG["seed"])

def load_env_file():
    # Lee el .env del proyecto (esta carpeta o una padre) sin pisar variables ya definidas
    for d in [Path.cwd(), *Path.cwd().parents]:
        f = d / ".env"
        if f.exists():
            for ln in f.read_text(encoding="utf-8").splitlines():
                if "=" in ln and not ln.lstrip().startswith("#"):
                    k, v = ln.split("=", 1)
                    os.environ.setdefault(k.strip(), v.strip().strip('"').strip("'"))
            return f
    return None
ENV_FILE = load_env_file()
print("Variables leídas de:", ENV_FILE or "ninguno (se usan variables de entorno)")

def get_secret(name):
    k = os.environ.get(name)
    if k: return k
    try:
        from kaggle_secrets import UserSecretsClient
        return UserSecretsClient().get_secret(name)
    except Exception: pass
    try:
        from google.colab import userdata
        return userdata.get(name)
    except Exception: pass
    raise RuntimeError(f"Falta el secreto/variable {name}.")
print("Salida en:", OUT.resolve())

## 1. Clases del dataset

In [ ]:
def find_classes():
    if CFG["classes_json"]: return Path(CFG["classes_json"])
    c = [Path("classes.json"), Path("food_prep/classes.json"), Path("/kaggle/working/food_prep/classes.json"), Path("/content/food_prep/classes.json")]
    k = Path("/kaggle/input")
    if k.exists(): c += list(k.glob("*/classes.json")) + list(k.glob("*/*/classes.json"))
    for p in c:
        if p.exists(): return p
    raise FileNotFoundError("No encuentro classes.json: pon su ruta en CFG['classes_json'].")

CLS = json.load(open(find_classes(), encoding="utf-8"))
if CFG["limit"]: CLS = CLS[:CFG["limit"]]
def human(s): return re.sub(r"\s+", " ", str(s).replace("-", " ").replace("_", " ")).strip()
print(f"{len(CLS)} clases" + (" (modo prueba)" if CFG["limit"] else " (esperadas 323)"))
for c in CLS[:5]: print("  ", c["idx"], c["name"], "| agrupa:", c.get("old_names", [])[:3])

## 2. Clientes: caché, LLM (OpenCode Go) y API de USDA

Todo se guarda en `llm_cache.jsonl` y `usda_cache.jsonl`. Si se agota el límite del LLM o la cuota de USDA, basta relanzar.

In [ ]:
class Cache:
    def __init__(self, path):
        self.path, self.d, self.lock = Path(path), {}, threading.Lock()
        if self.path.exists():
            for ln in open(self.path, encoding="utf-8"):
                try: o = json.loads(ln); self.d[o["k"]] = o["v"]
                except Exception: pass
    def put(self, k, v):
        with self.lock:
            self.d[k] = v
            with open(self.path, "a", encoding="utf-8") as f: f.write(json.dumps({"k": k, "v": v}, ensure_ascii=False) + "\n")
LLM_CACHE, API_CACHE = Cache(OUT / "llm_cache.jsonl"), Cache(OUT / "usda_cache.jsonl")

# ---------------- LLM ----------------
LLM_KEY = get_secret(CFG["api_key_env"])
import uuid
SESSION = str(uuid.uuid4())   # OpenCode Go exige x-opencode-session (una sesión estable por ejecución)
LHEAD = {"Authorization": f"Bearer {LLM_KEY}", "Content-Type": "application/json",
         "x-opencode-session": SESSION, "User-Agent": "calorie-counter-nutri-map/1.0"}
try:
    ids = [m["id"] for m in requests.get(CFG["models_url"], headers=LHEAD, timeout=30).json().get("data", [])]
    print("Modelos disponibles en tu plan:", ids)
    for m in [x for x in (CFG["model_a"], CFG["model_b"]) if x]:
        if ids and m not in ids: print(f"⚠️ '{m}' no aparece en la lista: cambia CFG['model_a'/'model_b'].")
except Exception as e:
    print("No pude listar modelos:", type(e).__name__, e)

class LLM:
    def __init__(self, model): self.model = model
    def chat(self, system, user):
        body = dict(model=self.model, temperature=CFG["temperature"], max_tokens=CFG["max_tokens"],
                    messages=[{"role": "system", "content": system}, {"role": "user", "content": user}])
        err = None
        for att in range(6):
            try:
                r = requests.post(CFG["base_url"], headers=LHEAD, json=body, timeout=CFG["timeout"])
                if r.status_code in (429, 500, 502, 503, 504): time.sleep(2 ** att * 2); continue
                if r.status_code == 400 and "temperature" in body and "temperature" in r.text.lower():
                    body.pop("temperature"); continue
                r.raise_for_status()
                return r.json()["choices"][0]["message"].get("content") or ""
            except requests.RequestException as e:
                err = e; time.sleep(2 ** att)
        raise RuntimeError(f"{self.model}: falló tras reintentos ({err})")

def parse_json(txt):
    if not txt: return None
    t = re.sub(r"<think>.*?</think>", "", txt, flags=re.S)
    t = re.sub(r"```(?:json)?", "", t)
    a, b = t.find("{"), t.rfind("}")
    if a < 0 or b <= a: return None
    try: return json.loads(t[a:b + 1])
    except json.JSONDecodeError: return None

def run_stage(llm, stage, items):
    def work(it):
        idx, s, u = it
        key = f"{llm.model}|{stage}|{idx}|{hashlib.md5((s + u).encode()).hexdigest()[:10]}"
        if key in LLM_CACHE.d: return idx, LLM_CACHE.d[key]
        js = None
        for _ in range(3):
            try: js = parse_json(llm.chat(s, u))
            except RuntimeError as e:
                print(f"   ⚠️ {llm.model} · {stage} · clase {idx}: {e}"); break
            if js is not None: break
        if js is not None: LLM_CACHE.put(key, js)
        return idx, js
    with ThreadPoolExecutor(CFG["workers"]) as ex:
        res = dict(tqdm(ex.map(work, items), total=len(items), desc=f"{stage} · {llm.model}"))
    print(f"   {stage}/{llm.model}: {sum(v is None for v in res.values())} respuestas sin JSON válido")
    return res
LA = LLM(CFG["model_a"])
LB = LLM(CFG["model_b"]) if CFG["model_b"] else None

# ---------------- USDA FoodData Central ----------------
class QuotaStop(RuntimeError): pass
class FDC:
    def __init__(self, key): self.key, self.low, self.calls = key, False, 0
    def call(self, method, path, body=None):
        if self.low: raise QuotaStop("Cuota de USDA casi agotada: relanza esta celda dentro de ~1 hora (la caché conserva el avance).")
        url = CFG["usda_base"] + path
        for att in range(5):
            try:
                r = (requests.post(url, params={"api_key": self.key}, json=body, timeout=60) if method == "POST"
                     else requests.get(url, params={"api_key": self.key}, timeout=60))
            except requests.RequestException:
                time.sleep(2 ** att); continue
            if r.status_code == 429: self.low = True; raise QuotaStop("USDA respondió 429 (límite alcanzado): espera ~1 hora y relanza.")
            if r.status_code in (500, 502, 503, 504): time.sleep(2 ** att * 2); continue
            r.raise_for_status(); self.calls += 1
            rem = r.headers.get("X-RateLimit-Remaining")
            if rem is not None and rem.isdigit() and int(rem) < CFG["quota_reserve"]: self.low = True
            return r.json()
        raise RuntimeError(f"USDA {path}: falló tras reintentos")
API = FDC(get_secret(CFG["usda_key_env"]))

def parse_nutrients(item):
    # Acepta los tres formatos de la API: búsqueda (nutrientId/value), abridged (number/amount) y full (nutrient{}/amount)
    out = []
    for n in item.get("foodNutrients", []) or []:
        if isinstance(n.get("nutrient"), dict):
            nu = n["nutrient"]; nid, num, name, unit, val = nu.get("id"), nu.get("number"), nu.get("name"), nu.get("unitName"), n.get("amount")
        else:
            nid, num = n.get("nutrientId"), n.get("nutrientNumber", n.get("number"))
            name, unit, val = n.get("nutrientName", n.get("name")), n.get("unitName"), n.get("value", n.get("amount"))
        if val is None or name is None: continue
        out.append(dict(nutrient_id=nid, number=None if num is None else str(num), name=name, unit=(unit or "").lower(), amount=float(val)))
    return out

def energy_kcal(nuts):
    for num in ("208", "957", "958"):        # energía (kcal); 957/958 = Atwater (Foundation)
        for n in nuts:
            if n["number"] == num and n["unit"] == "kcal": return n["amount"]
    return None

def fdc_search(q):
    key = f"search|{q.lower()}|{'/'.join(CFG['data_types'])}|{CFG['page_size']}"
    if key in API_CACHE.d: return API_CACHE.d[key]
    js = API.call("POST", "/foods/search", {"query": q, "dataType": CFG["data_types"], "pageSize": CFG["page_size"]})
    foods = [dict(fdc_id=str(f["fdcId"]), description=f.get("description", ""), data_type=f.get("dataType", ""),
                  category=f.get("foodCategory", ""), kcal=energy_kcal(parse_nutrients(f))) for f in js.get("foods", [])]
    API_CACHE.put(key, foods); return foods

## 3. Expansión de consultas (modelo A)

In [ ]:
SYS_EXPAND = '''Eres nutricionista experto en tablas de composición de alimentos (USDA FoodData Central).
Recibes el nombre de una clase de un dataset de reconocimiento de comida (nombres en inglés tomados de una base de datos suiza; pueden ser genéricos o muy específicos).
Devuelve SOLO un objeto JSON con estas claves:
"kind": "ingredient" | "dish" | "product" | "beverage",
"state": "raw" | "cooked" | "dried" | "processed" | "unknown",
"en_query": consulta corta en inglés al estilo USDA (p. ej. "Cabbage, savoy, cooked"),
"alt_en": hasta 3 sinónimos o alternativas en inglés, de la más a la menos probable (p. ej. nombres americanos del mismo alimento).'''

def user_expand(c): return f"Clase: {human(c['name'])}\nNombres originales agrupados en esta clase: {[human(n) for n in c.get('old_names', [])][:8]}"
EXP = run_stage(LA, "expand", [(c["idx"], SYS_EXPAND, user_expand(c)) for c in CLS])
def exp_of(c):
    e = EXP.get(c["idx"]) or {}
    return dict(kind=e.get("kind", "unknown"), state=e.get("state", "unknown"),
                en=[q for q in [e.get("en_query")] + list(e.get("alt_en") or []) if q] or [human(c["name"])])
for c in CLS[:4]: print(human(c["name"]), "→", exp_of(c))

## 4. Candidatos desde la API de USDA

Por clase: la consulta principal + la mejor alternativa (+ el nombre original si no hubo resultados). Los resultados se mezclan intercalados y se quitan duplicados. Se ve el avance y, si la cuota se agota, la celda se detiene con un mensaje claro y conserva todo lo ya consultado.

In [ ]:
def candidates_for(c):
    e = exp_of(c); qs = list(dict.fromkeys(e["en"][:CFG["queries_per_class"]] + [human(c["name"])]))
    lists = [fdc_search(q) for q in qs]
    if not any(lists): lists = [fdc_search(human(c["name"]))]
    out, seen = [], set()
    for rank in range(max(map(len, lists), default=0)):
        for L in lists:
            if rank < len(L) and L[rank]["fdc_id"] not in seen:
                seen.add(L[rank]["fdc_id"]); out.append(L[rank])
    return out[:CFG["n_candidates"]]

CAND = {}
try:
    for c in tqdm(CLS, desc="búsquedas USDA"): CAND[c["idx"]] = candidates_for(c)
except QuotaStop as e:
    print("⛔", e, f"| clases con candidatos: {len(CAND)}/{len(CLS)}")
    raise
print(f"Llamadas a la API en esta sesión: {API.calls} (la caché evita repetir las ya hechas)")
POOL = {f["fdc_id"]: f for L in CAND.values() for f in L}
empty = [c["name"] for c in CLS if not CAND[c["idx"]]]
print(f"Clases sin ningún candidato: {len(empty)}", empty[:8])
def fmt(f): return f"{f['fdc_id']} | {f['description']} | {f['data_type']}" + (f" | {f['kcal']:.0f} kcal/100 g" if f["kcal"] is not None else "")

## 5. Selección (modelos A y B) — solo entre candidatos

Si un id no está en la lista de candidatos, se descarta y se marca `id_invalido`.

In [ ]:
SYS_SELECT = '''Eres nutricionista experto en composición de alimentos. Debes emparejar una clase de comida con UNA entrada de USDA FoodData Central, eligiendo SOLO entre los candidatos dados. Usa el id exacto del candidato (solo el número). Nunca inventes ids.
Reglas:
- Respeta el estado del alimento (crudo, cocido, seco, frito...). Si la clase no lo indica, elige el estado en que normalmente se come.
- Para ingredientes simples prefiere Foundation o SR Legacy; para platos preparados o recetas, Survey (FNDDS).
- "match": "exact" (mismo alimento y estado) | "close" (misma preparación o variedad muy parecida) | "proxy" (alimento distinto pero nutricionalmente parecido) | "composite" (plato o producto compuesto sin entrada fiel; elige la más cercana o null) | "none" (ningún candidato sirve; id null).
- Ante duda entre dos candidatos válidos, elige el más genérico.
- Categoría genérica (fish, meat, cheese, nuts...) sin preparación indicada: elige la preparación típica de consumo.
- Un producto de marca o regional sin equivalente: usa "proxy" con el candidato más parecido, o "none".
Responde SOLO con JSON: {"id":"123456|null","match":"...","state_ok":true|false,"reason":"máx. 25 palabras"}
"state_ok" es false si la entrada elegida no respeta el estado del alimento.'''

def user_select(c):
    e = exp_of(c)
    return (f"Clase: {human(c['name'])}\nNombres originales agrupados: {[human(n) for n in c.get('old_names', [])][:8]}\n"
            f"Tipo estimado: {e['kind']} | estado estimado: {e['state']}\n\nCANDIDATOS USDA:\n" + ("\n".join(fmt(f) for f in CAND[c["idx"]]) or "(ninguno)"))

def norm_sel(js, c):
    out = dict(id=None, match="none", invalid=False, state_ok=True, reason="", failed=js is None)
    if js is None: out.update(state_ok=False, reason="sin respuesta válida"); return out
    i = js.get("id"); m = str(js.get("match", "none")).lower()
    out["state_ok"] = bool(js.get("state_ok", True)); out["reason"] = str(js.get("reason", ""))[:200]
    i = None if i is None else re.sub(r"^[Uu]:", "", str(i).strip())
    if i in (None, "", "null", "None"): i = None
    elif i not in {f["fdc_id"] for f in CAND[c["idx"]]}: out["invalid"] = True; i = None; m = "none"
    m = m if m in ("exact", "close", "proxy", "composite", "none") else "none"
    if i is None and m != "composite": m = "none"       # 'composite' sin id se conserva: el plato necesita decisión (receta/proxy)
    out["id"], out["match"] = i, m
    return out

SEL = {}
items = [(c["idx"], SYS_SELECT, user_select(c)) for c in CLS]
for tag, llm in [t for t in (("A", LA), ("B", LB)) if t[1]]:
    raw = run_stage(llm, "select", items)
    SEL[tag] = {c["idx"]: norm_sel(raw[c["idx"]], c) for c in CLS}

## 6. Combinar, marcar y decidir qué se revisa

Estados: **`auto_ok`** (ambos coinciden y es `exact`/`close`, sin alertas), **`no_match`** (ambos coinciden en que no hay entrada) y **`review`** (cualquier alerta: `desacuerdo`, `proxy`, `composite`, `estado`, `id_invalido`, `sin_respuesta`, `contradiccion` (dice `none` y aun así elige id).

In [ ]:
RANK = {"exact": 0, "close": 1, "proxy": 2, "composite": 3, "none": 4}
def dsc(i): return POOL[i]["description"] if i in POOL else ""
REVISAR_TODO = len(SEL) == 1          # con un solo modelo, nada se acepta sin revisión humana

def combine(c):
    i = c["idx"]; sels = [SEL[t][i] for t in SEL]; a = sels[0]
    b = SEL["B"][i] if "B" in SEL else dict(id=None, match="none", invalid=False, state_ok=True, reason="", failed=False)
    flags = []
    agree = len({s["id"] for s in sels}) == 1
    worst = max((s["match"] for s in sels), key=lambda m: RANK[m])
    if not agree: flags.append("desacuerdo")
    elif worst in ("proxy", "composite"): flags.append(worst)
    if any(s["invalid"] for s in sels): flags.append("id_invalido")
    if not all(s["state_ok"] for s in sels): flags.append("estado")
    if any(s["failed"] for s in sels): flags.append("sin_respuesta")
    if any(s["match"] == "none" and s["id"] for s in sels): flags.append("contradiccion")
    draft = (a["id"] or "") if agree else ""
    st = "review" if (flags or REVISAR_TODO) else ("no_match" if not draft else "auto_ok")
    return dict(idx=i, clase=c["name"], agrupa=" | ".join(c.get("old_names", [])[:6]), status=st, flags=";".join(sorted(set(flags))),
                usda_draft=draft, usda_draft_desc=dsc(draft), usda_kcal=POOL[draft]["kcal"] if draft in POOL else "",
                usda_A=a["id"] or "", usda_A_desc=dsc(a["id"] or ""), usda_B=b["id"] or "", usda_B_desc=dsc(b["id"] or ""),
                usda_match=worst if agree else "disputed", razon_A=a["reason"], razon_B=b["reason"])

DF = pd.DataFrame([combine(c) for c in CLS])
print(DF["status"].value_counts().to_string())
print("\nAlertas más frecuentes:", Counter(f for s in DF["flags"] for f in s.split(";") if f).most_common(8))

## 7. Cola de revisión

`review_queue.csv` contiene todo lo marcado más una **muestra aleatoria de auditoría** (`audit`) de lo que pasó solo. Ábrelo en Excel/Sheets, rellena `decision` y guárdalo como `review_queue_done.csv`:

- **`ok`** → aceptas el borrador (`usda_draft`).
- **`cambiar`** → escribes el FDC id correcto en `final_usda` (puede ser el de A o el de B de la fila, o cualquier id de <https://fdc.nal.usda.gov>).
- **`sin_match`** → la clase no tiene entrada válida.
- **`receta`** → plato compuesto: se calcula con las filas de `recetas.csv` (celda 8).

En las filas `review` con desacuerdo el borrador va vacío: normalmente usarás `cambiar` con el id de A o de B (sus descripciones están en la fila).

In [ ]:
rng = np.random.RandomState(CFG["seed"])
passed = DF.index[DF["status"].isin(["auto_ok", "no_match"])].to_numpy()
n_aud = int(math.ceil(CFG["audit_frac"] * len(passed))) if len(passed) else 0
aud = set(rng.choice(passed, n_aud, replace=False).tolist()) if n_aud else set()
DF["status_final"] = [("audit" if i in aud else s) for i, s in zip(DF.index, DF["status"])]

cols = ["idx", "clase", "agrupa", "status_final", "flags", "usda_draft", "usda_draft_desc", "usda_kcal", "usda_match",
        "usda_A", "usda_A_desc", "usda_B", "usda_B_desc", "razon_A", "razon_B"]
Q = DF[DF["status_final"].isin(["review", "audit"])][cols].copy()
Q["decision"] = ""; Q["final_usda"] = ""; Q["nota"] = ""
Q = Q.sort_values(["status_final", "idx"], ascending=[False, True])   # primero review, luego audit
Q.to_csv(OUT / "review_queue.csv", index=False, encoding="utf-8-sig")
DF.to_csv(OUT / "mapping_draft.csv", index=False, encoding="utf-8-sig")
print(DF["status_final"].value_counts().to_string())
print(f"\n→ {len(Q)} filas para revisar a mano de {len(DF)} clases: {OUT / 'review_queue.csv'}")

## 8. Tabla final con todos los nutrientes

Tras revisar, guarda `review_queue_done.csv` y ejecuta esta celda. Descarga los nutrientes **completos** de los ids finales (`POST /foods`, 20 por llamada, ~17 llamadas) y genera:

- `nutrition_wide.csv`: una fila por clase, una columna por nutriente (por 100 g). Vacío = ese nutriente no se midió para esa entrada.
- `nutrition_long.csv`: formato largo (clase, fdc_id, nutriente, unidad, cantidad).
- `nutrient_dictionary.csv`: id, número, nombre y unidad de cada nutriente.
- `nutrition_map.json`: por clase, ids, nutrientes y porciones caseras (`gramWeight`), útiles para pasar de volumen/medida a gramos.

Además estima la **tasa de error de lo que no se revisó** con la muestra de auditoría (intervalo de Wilson al 95 %).

In [ ]:
def wilson(k, n, z=1.96):
    if n == 0: return (np.nan, np.nan)
    p = k / n; d = 1 + z * z / n; c = p + z * z / (2 * n); h = z * math.sqrt(p * (1 - p) / n + z * z / (4 * n * n))
    return ((c - h) / d, (c + h) / d)

def fetch_details(ids):
    ids = sorted({str(i) for i in ids if i}); got = {}
    for k in range(0, len(ids), 20):
        chunk = ids[k:k + 20]; key = "foods|full|" + ",".join(chunk)
        if key not in API_CACHE.d:
            js = API.call("POST", "/foods", {"fdcIds": [int(i) for i in chunk], "format": "full"})
            API_CACHE.put(key, js if isinstance(js, list) else js.get("foods", []))
        for f in API_CACHE.d[key]: got[str(f["fdcId"])] = f
    return got

def load_recipes():
    # recetas.csv: una fila por ingrediente; gramos = gramos de ese ingrediente (tal como se consume) por 100 g de plato listo
    p = Path(CFG["recipes_csv"])
    if not p.exists(): return {}
    R = pd.read_csv(p, dtype=str).fillna("")
    out = {}
    for idx, g in R.groupby("idx"):
        ings = [dict(nombre=r["ingrediente"], fdc_id=re.sub(r"\D", "", r["fdc_id"]), g=float(r["gramos_100g_plato"] or 0)) for _, r in g.iterrows()]
        total = sum(i["g"] for i in ings)
        if abs(total - 100) > 5: print(f"⚠️ receta {idx}: los ingredientes suman {total:.0f} g (deben sumar ~100 g por 100 g de plato listo)")
        out[int(idx)] = dict(ings=ings, porcion_g=next((float(x) for x in g["porcion_g"] if x), None))
    return out

def _energia_208(nuts):
    # entradas Foundation solo traen energia Atwater (957/958): se toma como 208 para que la suma no la cuente como 0
    if any(n["number"] == "208" for n in nuts): return nuts
    for num in ("957", "958"):
        for n in nuts:
            if n["number"] == num and n["unit"] == "kcal":
                return nuts + [dict(n, number="208", nutrient_id=1008, name="Energy")]
    return nuts

def recipe_profile(rec, det):
    # suma ponderada por gramos. Si un ingrediente no reporta un nutriente (p. ej. azúcares en un aceite), cuenta como 0
    per = []
    for ing in rec["ings"]:
        f = det.get(ing["fdc_id"]) if ing["fdc_id"] else None
        if f is None: raise ValueError(f"ingrediente '{ing['nombre']}' sin id válido ({ing['fdc_id'] or 'vacío'})")
        per.append({(n["nutrient_id"], n["number"]): n for n in _energia_208(parse_nutrients(f))})
    allk = set().union(*map(set, per)) if per else set()
    faltan = sum(1 for p in per for k in allk if k not in p)
    if faltan: print(f"   ℹ️ receta: {faltan} dato(s) de nutriente ausentes en algún ingrediente, contados como 0")
    out = []
    for k in sorted(allk, key=lambda k: str(k[1])):
        amt = sum((p[k]["amount"] if k in p else 0) * ing["g"] / 100 for p, ing in zip(per, rec["ings"]))
        base = next(p[k] for p in per if k in p)
        out.append(dict(base, amount=round(amt, 4)))
    return out

def finalize(path=None):
    path = Path(path or OUT / CFG["reviewed_csv"])
    rev = pd.read_csv(path, dtype=str).fillna("") if path.exists() else pd.DataFrame(columns=["idx", "decision"])
    if not path.exists(): print(f"⚠️ No encuentro {path}: se usa el borrador sin revisión humana.") if not path.exists() else print(f"Decisiones leídas de: {path}")
    R = {int(r["idx"]): r for _, r in rev.iterrows()}
    REC = load_recipes()
    rows, problemas, pendientes, aud_n, aud_err = [], [], 0, 0, 0
    for _, d in DF.iterrows():
        i = int(d["idx"]); r = R.get(i); dec = r["decision"].strip().lower() if r is not None else ""
        fid, match = d["usda_draft"], d["usda_match"]
        if dec not in ("", "ok", "cambiar", "sin_match", "receta"):
            problemas.append(f"{i} {d['clase']}: decisión desconocida '{dec}'")
        elif dec == "cambiar":
            fid, match = re.sub(r"\D", "", r["final_usda"]), "manual"
            if not fid: problemas.append(f"{i} {d['clase']}: 'cambiar' sin final_usda")
        elif dec == "sin_match": fid, match = "", "none"
        elif dec == "receta":
            fid, match = "", "receta"
            if i not in REC: problemas.append(f"{i} {d['clase']}: 'receta' pero no hay filas para esa clase en {CFG['recipes_csv']}")
        elif dec == "ok" and not fid and d["status_final"] != "no_match":
            problemas.append(f"{i} {d['clase']}: 'ok' sobre un borrador vacío (usa 'cambiar', 'receta' o 'sin_match')")
        elif not dec and d["status_final"] == "review": pendientes += 1
        if d["status_final"] == "audit" and dec in ("ok", "cambiar", "sin_match", "receta"):
            aud_n += 1; aud_err += (fid != d["usda_draft"]) or dec == "receta"
        rows.append(dict(idx=i, clase=d["clase"], status=d["status_final"], reviewed=bool(dec), flags=d["flags"], fdc_id=fid, match=match))
    if problemas:
        print(f"⚠️ {len(problemas)} decisiones incompletas o inválidas (esas filas quedan sin id):")
        for p in problemas[:30]: print("   -", p)
    M = pd.DataFrame(rows)
    used = {r["idx"]: REC[r["idx"]] for r in rows if r["match"] == "receta" and r["idx"] in REC}
    det = fetch_details(list(M["fdc_id"]) + [g["fdc_id"] for rec in used.values() for g in rec["ings"]])
    long_rows, wide_rows, js_out, nd = [], [], [], {}
    for _, m in M.iterrows():
        i = int(m["idx"])
        if i in used:
            rec = used[i]
            try: nuts = recipe_profile(rec, det)
            except ValueError as e: print(f"⚠️ receta {i} ({m['clase']}): {e}"); nuts = []
            fid_lab, extra = "RECETA", dict(ingredientes=rec["ings"])
            info = dict(fdc_description=f"RECETA ({len(rec['ings'])} ingredientes)", fdc_data_type="receta", kcal_100g=energy_kcal(nuts))
            portions = [dict(amount=1, gram_weight=rec["porcion_g"], modifier=None, description="porción típica del plato", unit="g")] if rec["porcion_g"] else []
        else:
            f = det.get(m["fdc_id"]) if m["fdc_id"] else None
            if m["fdc_id"] and f is None: print(f"⚠️ la API no devolvió el id {m['fdc_id']} (clase {m['clase']})")
            nuts = parse_nutrients(f) if f else []
            fid_lab, extra = m["fdc_id"], {}
            info = dict(fdc_description=(f or {}).get("description", ""), fdc_data_type=(f or {}).get("dataType", ""), kcal_100g=energy_kcal(nuts))
            portions = [dict(amount=p.get("amount"), gram_weight=p.get("gramWeight"), modifier=p.get("modifier"), description=p.get("portionDescription"),
                             unit=(p.get("measureUnit") or {}).get("name")) for p in (f or {}).get("foodPortions", [])]
        for n in nuts:
            nd[(n["nutrient_id"], n["number"])] = (n["name"], n["unit"])
            long_rows.append(dict(idx=i, clase=m["clase"], fdc_id=fid_lab, **n))
        js_out.append(dict(m.to_dict(), **info, nutrients={f"{n['name']} ({n['unit']})": n["amount"] for n in nuts}, portions=portions, **extra))
        wide_rows.append((m.to_dict(), info, nuts))
    L = pd.DataFrame(long_rows)
    # columnas del formato ancho: "Nombre (unidad)"; si dos nutrientes colisionan, se añade su número
    lab = {}
    for (nid, num), (name, unit) in nd.items(): lab[(nid, num)] = f"{name} ({unit})"
    cnt = Counter(lab.values())
    lab = {k: (v if cnt[v] == 1 else f"{v} [{k[1]}]") for k, v in lab.items()}
    W = []
    for m, info, nuts in wide_rows:
        row = {**m, **info}; row.update({lab[(n["nutrient_id"], n["number"])]: n["amount"] for n in nuts}); W.append(row)
    W = pd.DataFrame(W)
    W.to_csv(OUT / "nutrition_wide.csv", index=False, encoding="utf-8-sig")
    L.to_csv(OUT / "nutrition_long.csv", index=False, encoding="utf-8-sig")
    pd.DataFrame([dict(nutrient_id=k[0], number=k[1], name=v[0], unit=v[1], column=lab[k]) for k, v in nd.items()],
                 columns=["nutrient_id", "number", "name", "unit", "column"]).sort_values("number").to_csv(OUT / "nutrient_dictionary.csv", index=False)
    json.dump(js_out, open(OUT / "nutrition_map.json", "w", encoding="utf-8"), ensure_ascii=False, indent=1)
    print(f"Clases: {len(M)} | con entrada USDA: {(M['fdc_id'] != '').sum()} | recetas: {len(used)} | nutrientes distintos: {len(nd)} | filas 'review' sin decidir: {pendientes}" + ("  ⚠️ revísalas antes de usar el mapeo" if pendientes else ""))
    if len(W): print("Tipos de dato usados:", W["fdc_data_type"].value_counts().to_dict())
    if aud_n:
        lo, hi = wilson(aud_err, aud_n)
        print(f"Auditoría: {aud_err} diferencias en {aud_n} filas → tasa {aud_err/aud_n:.1%} (IC95 % {lo:.1%}–{hi:.1%}) para lo que no se revisó a mano")
    else: print("Aún no hay filas de auditoría revisadas: no se puede estimar la tasa de error.")
    return W
FIN = finalize()

### Notas para el informe y para el uso posterior
- **Reproducibilidad:** la API devuelve la versión vigente de cada base. Las tablas finales ya contienen los valores usados, y `usda_cache.jsonl` guarda las respuestas crudas; archívalos junto con la fecha de ejecución. Guarda también `CFG` (modelos, `temperature=0`, tipos de dato, semilla de la auditoría).
- **Por 100 g:** los valores de Foundation, SR Legacy y FNDDS están expresados por 100 g de porción comestible; `Branded` se excluyó a propósito (viene por porción y con calidad variable).
- **Cobertura desigual de nutrientes:** cada tipo de dato mide un conjunto distinto (FNDDS tiene menos nutrientes que Foundation/SR Legacy), así que habrá columnas vacías. `fdc_data_type` queda en la tabla para saberlo.
- **Platos compuestos (`match` = `receta`):** se calculan sumando los nutrientes de `recetas.csv` (gramos por 100 g de plato listo, cada ingrediente con id de USDA). Solo se conservan los nutrientes presentes en todos los ingredientes.
- Si agregas más clases o cambias el prompt, la caché se invalida solo para lo que cambió (la clave incluye un hash del prompt).